# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamedkhaled600/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

***Answer:***

**One row = one content item, on one client, on one calendar day** — the daily-performance grain
(`report_date x client_hash_id x content_hash_id`) in `fact_content_daily_performance`.

**Tables I'll use:**
- `fact_content_daily_performance` (partitioned by month) — the daily GSC/GA4 metrics, my main
  table.
- `dim_content` — static content attributes (content_type, word_count, first-seen date) to join in
  by `content_hash_id`.
- `dim_clients` — to read `gsc_data_start` / `ga4_data_start` per client, since history depth
  differs wildly by client and I need to filter on it rather than assume one global window.

**Time window:** developing everything on the **mid-panel month `month=2026-03`**, per the
iteration rule — the `_sample` table is June 2026, the panel's final month, so it's the natural
outcome window of any past→future label and off-limits for label logic (query-mechanics testing
only). I treat June 2026 as a sealed test month I don't touch yet.

**What I'd predict or rank (label/proxy):** whether a content item is declining within the month
— comparing its performance in the second half of the month to the first half (same idea as
`is_declining_label` from the starter CSV, now computed fresh from the warehouse's own daily
rows instead of a delivered column).

**One thing I deliberately exclude:** rows where `dim_content.is_deleted = TRUE`. A deleted
page has no editor action available — you can't "refresh" or "prioritize" something that no
longer exists — so scoring it would produce a recommendation nobody can act on. (I confirmed by
schema discovery that no health-score/flag-style product column ships in these three tables, so
the FlyRank "never use a product flag as a feature" rule has nothing to exclude here — worth
noting as a difference from the starter CSV, which does carry that kind of column.)

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install duckdb
import duckdb

con = duckdb.connect()

# Register the HF read token as a DuckDB secret — read from Colab Secrets, NEVER pasted in a cell
# (this repo is public). In Colab: key icon in the left sidebar -> add secret named HF_TOKEN.
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "fact_content_daily_performance/month=2026-03/*.parquet"

# Schema discovery FIRST — column names below are my best reading of the flyrank-data skill doc;
# confirm them here before trusting the rest of this notebook, and swap any that differ.
print("--- fact_content_daily_performance columns ---")
print(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{BASE}/{MONTH}') LIMIT 1").df())
print()
print("--- dim_content columns ---")
print(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{BASE}/dim_content.parquet') LIMIT 1").df())
print()
print("--- dim_clients columns ---")
print(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{BASE}/dim_clients.parquet') LIMIT 1").df())


--- fact_content_daily_performance columns ---
                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users 

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

***Answer:***

- **Feature** (knowable before the prediction moment): prior-window rolling clicks/impressions
  (e.g. trailing 7-day sums using only days before `report_date`), `gsc_avg_position` **lagged by
  one day**, static content attributes from `dim_content` (content_type, word_count,
  content_age_days at the report date).
- **Label / proxy** (never a feature): the within-month decline flag I compute — second-half vs
  first-half performance for the same content item — and anything the label is directly derived
  from (this month's own totals), same trap as `is_declining_label` in the starter CSV.
- **Context** (for joining/splitting/filtering, never for the model to learn from): `content_hash_id`,
  `client_hash_id`, `report_date` itself, `month` partition key.
- **Excluded:** GA4 columns on rows where `ga4_data_available IS NOT TRUE` (zero there means "not
  measured yet," not "zero engagement" — including them as real zeros would inject a fake signal);
  content rows where `dim_content.is_deleted = TRUE` (no action is possible on a deleted page); and
  the AI-referrer breakdown columns (`ai_chatgpt`, `ai_perplexity`, etc.) — real and usable, just
  out of scope for a refresh-priority lane, not a privacy or leakage concern.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

***Answer:***

Three required queries below (grain, counts + date span, availability with `IS TRUE`), then the
five-feature frame, then the deliberate-leak trap.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# --- Query 1: grain check ---
# Claim: one row = one (report_date, client_hash_id, content_hash_id). Zero rows back below proves it.
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM read_parquet('{BASE}/{MONTH}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()

print("Rows violating the stated grain (should be empty):")
print(grain_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating the stated grain (should be empty):
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []


### Query 2: row count + date span

*Claim: `month=2026-03` gives me one calendar month of rows for a subset of the ~104 clients and
~520k content items in the panel — I'll compare the actual count/date span against that
expectation.*

In [3]:
# --- Query 2: row count + date span ---
counts = con.sql(f"""
    SELECT
        COUNT(*)                    AS n_rows,
        MIN(report_date)            AS min_date,
        MAX(report_date)            AS max_date,
        COUNT(DISTINCT client_hash_id)   AS n_clients,
        COUNT(DISTINCT content_hash_id)  AS n_content
    FROM read_parquet('{BASE}/{MONTH}')
""").df()

print(counts)


    n_rows   min_date   max_date  n_clients  n_content
0  9841378 2026-03-01 2026-03-31         55     331437


### Query 3: availability, filtered with `IS TRUE`

*Claim: a meaningful share of rows have `ga4_data_available IS NOT TRUE` (client hasn't started
GA4 history yet at that date) — zeros there are missing data, not real zeros, so I check how many
rows actually survive an `IS TRUE` filter before I'd ever trust a GA4-based feature.*

**Measured:** only 413,966 of 9,841,378 rows (**4.2%**) have `ga4_data_available IS TRUE` in this
month — far sparser than I expected. GA4 is close to unusable as a feature source for
`month=2026-03`; this is the concrete reason my five features below are GSC-only.

In [4]:
# --- Query 3: availability check ---
availability = con.sql(f"""
    SELECT
        COUNT(*) AS n_total,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS n_ga4_available,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)
              / COUNT(*), 1) AS pct_ga4_available
    FROM read_parquet('{BASE}/{MONTH}')
""").df()

print(availability)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   n_total  n_ga4_available  pct_ga4_available
0  9841378         413966.0                4.2


### Five features (max) — each tagged "knowable at the decision moment because…"

Built as a small content-month aggregate (not a raw 30-row-per-item frame) — one row per
`(client_hash_id, content_hash_id)` for `month=2026-03`, using only the **first half of the month**
(days 1–15) to build features, so nothing here peeks at the second half I'll use for the label.

1. **`prior_clicks_h1`** — sum of clicks, days 1–15. *Knowable because it's fully in the past
   relative to any decision made on day 16.*
2. **`prior_impressions_h1`** — sum of impressions, days 1–15. *Same reasoning — pure history.*
3. **`prior_avg_position_h1`** — mean `gsc_avg_position`, days 1–15. *A lagged average of an
   already-observed ranking signal, not the current or future position.*
4. **`content_type`** (from `dim_content`) — *a static content attribute, fixed before the content
   ever earned a single impression — always knowable, with no time dimension at all.*
5. **`content_age_days_h1_end`** — `report_date - dim_content.content_created_date` measured at day 15.
   *Just elapsed time; knowable the instant we ask, never depends on a future date.*

In [5]:
# --- Five-feature frame (content-month aggregate, first-half-of-month only) ---
features = con.sql(f"""
    WITH h1 AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_clicks)      AS prior_clicks_h1,
               SUM(gsc_impressions) AS prior_impressions_h1,
               AVG(gsc_avg_position) AS prior_avg_position_h1
        FROM read_parquet('{BASE}/{MONTH}')
        WHERE report_date <= DATE '2026-03-15'
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT h1.*, dc.content_type,
           DATE '2026-03-15' - dc.content_created_date AS content_age_days_h1_end
    FROM h1
    JOIN read_parquet('{BASE}/dim_content.parquet') dc
        USING (client_hash_id, content_hash_id)
    WHERE dc.is_deleted = FALSE  -- contract exclusion: no editor action possible on deleted content
""").df()

print("Feature frame shape:", features.shape)
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (313271, 7)


,client_hash_id,content_hash_id,prior_clicks_h1,prior_impressions_h1,prior_avg_position_h1,content_type,content_age_days_h1_end
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,0.0,111.0,5.222776,keyword article,31
1,client_62f4a7e64f5e0096,content_67741cce996cfafa,1.0,38.0,4.638889,keyword article,31
2,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,1.0,219.0,3.737399,keyword article,31
3,client_62f4a7e64f5e0096,content_ac8663da7484669a,0.0,20.0,3.597222,keyword article,31
4,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,0.0,1494.0,6.156643,keyword article,31


### The trap: add one label-derived column on purpose

I'll build the label the same way as `is_declining_label` in the starter data: **declining = 1 if
second-half (days 16–end) clicks are lower than first-half clicks for that content item.** Then
I'll deliberately add `second_half_clicks` itself as a "feature" — the exact quantity the label is
built from — fit a quick logistic regression, and watch the score jump toward a perfect 1.0. That's
the leakage lesson from notebook 02, reproduced here on real warehouse data. Then I delete the leak
column and keep the honest, lower score.

**Measured:** leaky AUC = **1.000** — a perfect score is itself the red flag, since
`second_half_clicks` almost fully determines the label by construction. With the leak removed,
honest AUC = **0.828** — real, meaningfully-above-chance signal from prior-window features alone,
with no shortcut baked in.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

h2 = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS second_half_clicks
    FROM read_parquet('{BASE}/{MONTH}')
    WHERE report_date > DATE '2026-03-15'
    GROUP BY client_hash_id, content_hash_id
""").df()

panel = features.merge(h2, on=["client_hash_id", "content_hash_id"], how="inner")
panel["declining"] = (panel["second_half_clicks"] < panel["prior_clicks_h1"]).astype(int)

honest_X = panel[["prior_clicks_h1", "prior_impressions_h1", "prior_avg_position_h1",
                   "content_age_days_h1_end"]].fillna(0)
leaky_X  = honest_X.copy()
leaky_X["second_half_clicks"] = panel["second_half_clicks"]  # <-- the deliberate leak
y = panel["declining"]

def quick_auc(X, y, label):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
    model = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
    auc = roc_auc_score(yte, model.predict_proba(Xte)[:, 1])
    print(f"{label}: AUC = {auc:.3f}")
    return auc

print("--- WITH the leak (second_half_clicks included) ---")
quick_auc(leaky_X, y, "Leaky model")

print()
print("--- Leak removed — the honest number I actually keep ---")
quick_auc(honest_X, y, "Honest model")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- WITH the leak (second_half_clicks included) ---
Leaky model: AUC = 1.000

--- Leak removed — the honest number I actually keep ---
Honest model: AUC = 0.828


np.float64(0.8282189499140601)

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

***Answer:***

**Named limitation: GA4 coverage in this month is almost nonexistent — 4.2% of rows, measured in
Query 3 above — so this slice can only honestly support GSC-based (impressions/clicks/position)
signals, not engagement-based ones.** Any claim my model makes is really a claim about *search
visibility behavior*, not about on-page engagement — a page could have excellent GA4 engagement
the model never sees. History depth is also wildly uneven across clients more generally (per
`dim_clients.gsc_data_start`), so `month=2026-03` under-represents clients whose panel history
starts later — a model trained here will generalize worse to those clients. I did not attempt to
fix either gap by backfilling or dropping rows — I'm naming them as boundaries of what this slice
can honestly claim, not solving them in this notebook.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.